In [4]:
import random
import math

In [ ]:
class PerudoGame:

    def __init__(self, players = 2, dice_per_player = 5):
        
        self.players = []
        self.dice_per_player = dice_per_player
        
        for i in range(players):
            self.players.append(Player(f"Player {i}",dice_per_player))

        self.previous_bids = []
        self.current_player = None
        self.current_bid = None
        self.challenger = None
        self.next_start_player = None
        
    def reset(self):
        
        for player in self.players:
            player.reset()
        
        self.current_player = self.players[0]
        self.next_start_player = None
        self.current_bid = None
        self.challenger = None

        
# game loop functions
    def next_round(self):
        
        self.remove_players_with_no_dice()
        if self.check_is_winner():
            return
        
        self.current_player = self.next_start_player
        self.previous_bids = []
        self.players_roll()
    def players_roll(self):
        for player in self.players:
            player.roll()
    def next_player(self):
        current_index = self.players.index(self.current_player)
        self.current_player = self.players[(current_index + 1) % len(self.players)]
        #self.current_player.make_action
    def remove_players_with_no_dice(self):
        for player in self.players.copy():
            if len(player.dice) == 0:
                self.players.remove(player)
    def check_is_winner(self):
        if len(self.players) == 1:
            print(f"{self.player[0].name} has won")
            return True
        return False

  
# bid creation and testing functions   
    def is_bid_legal(self, Action):
        
        # if current bid is 1s:
        # at least double quantity and different face 
        # higher quantity and face is 1s
        if self.current_bid.face == 1:
            if (
                Action.quantity >= self.current_bid.quantity*2
                or (Action.quantity>self.current_bid.quantity and Action.face == 1)
            ):
                return True

        # normal bid:
        # raise either quantity or face
        elif (
            Action.quantity > self.current_bid.quantity 
            or Action.face > self.current_bid.face
            ): 

            return True

        # player can choose to halve quantity of the current bid if the face is 1s
        # provided that this bid has not already been made
        # (the half quantity has to be greater of 2 halves)
        elif (
            self.check_previous_bids(Action) 
            and (Action.quantity == (self.current_bid.quantity+1)//2 
            and Action.face == 1)
              ):
            
            return True
        return False    
    def check_previous_bids(self, Action):
        for previous_action in self.previous_bids:
            if previous_action.quantity == Action.quantity and previous_action.face == Action.face:
                return False
        return True
    def apply_bid(self, Action):
        self.current_bid = Action
        self.previous_bids.append(Action)
        pass

# calling bluff functions
    def count_face(self, face):
        count = 0
        for player in self.players:
            count += player.dice.count(face)
        return count
    def is_call_false(self):
        return self.count_face(self.current_bid.face)>=self.current_bid.quantity
    def resolve_call(self):
        challenger = self.current_player
        if self.is_call_false():
            self.next_start_player = challenger
            challenger.remove_die()
        else:
            self.next_start_player = self.current_bid.bidder
            self.current_bid.bidder.remove_die()
# perudo action functions
    def resolve_perudo(self):
        if self.count_face(self.current_bid.face) == self.current_bid.quantity:
            self.current_player.add_die()
        else:
            self.current_player.remove_die()


    def perform_action(self, action):
        if action.action_type == "bid":
            if self.is_bid_legal(action):
                self.apply_bid(action)
                self.next_player()
            else:
                print("illegal bid")
                #player prompted for another action
            
        elif action.action_type == "call_bluff":
            self.resolve_call()
            self.next_round()
        elif action.action_type == "perudo":
            self.resolve_perudo()
            self.next_round()
            pass

In [17]:
class Player:
    
    def __init__(self, name: str, initial_dice_quantity):
        
        self.name = name
        self.initial_dice_quantity = initial_dice_quantity
        self.dice = []
    def reset(self):
            self.dice = []
            for i in range(self.initial_dice_quantity):
                        self.dice.append(random.randint(1,6))


    # control functions
    def remove_die(self):
        self.dice.pop()
    def add_die(self):
        if len(self.dice) < self.initial_dice_quantity:
            self.dice.append(1)
    def roll(self):
        for i in range(len(self.dice)):
            self.dice[i] = random.randint(1,6)
    
    
    # turn actions
    def make_bid(self, quantity, face):
        return Action("bid", self, quantity, face)
    def call_bluff(self):
        return Action("call_bluff", self)
    def call_perudo(self):
        return Action("call_perudo", self)

In [ ]:
class Action:
    def __init__(self, action_type, player, quantity = None, face = None):
        self.action_type = action_type
        self.player = player
        self.quantity = quantity
        self.face = face

In [18]:
game = PerudoGame(players=2, dice_per_player=5)

game.reset()

while True:

    # Show current game state
    print("\n------------------------")
    
    for player in game.players:
        print(f"{player.name}: {player.dice}")

    print("------------------------")

    # Show current bid
    if game.current_bid is None:
        print("No current bid.")
    else:
        print(
            f"Current bid: "
            f"{game.current_bid.quantity} x {game.current_bid.face}s"
        )

    # Show whose turn it is
    print(f"\n{game.current_player.name}'s turn")

    # Get action
    action_type = input(
        "Choose action (bid / call / perudo): "
    ).lower()

    if action_type == "bid":

        quantity = int(input("Quantity: "))
        face = int(input("Face: "))

        action = Action(
            "bid",
            game.current_player,
            quantity,
            face
        )

    elif action_type == "call":

        action = Action(
            "call_bluff",
            game.current_player
        )

    elif action_type == "perudo":

        action = Action(
            "perudo",
            game.current_player
        )

    else:
        print("Invalid action.")
        continue

    # Give action to the game
    game.perform_action(action)


------------------------
Player 0: [1, 3, 5, 4, 3]
Player 1: [4, 6, 5, 6, 2]
------------------------
No current bid.

Player 0's turn


AttributeError: 'NoneType' object has no attribute 'face'